# 🧠 Advanced Hybrid & Graph RAG Laboratory

Welcome to the **Pedagogical Advanced RAG Laboratory**! This interactive notebook walks you step-by-step through building a production-grade, state-of-the-art Retrieval-Augmented Generation (RAG) system using **LlamaIndex** and **Ragas**.

### 🌟 Laboratory Architecture:
1. **Colab-Ready Setup**: Automated installation of required dependencies and API configuration.
2. **Dynamic Ingestion & Adaptive Parsing**: File uploads, classification based on content structure, and dynamic chunk selection.
3. **Dual-Index Storage Context**: Embedding documents into a SQLite ChromaDB vector index and building an entity-relation Knowledge Graph.
4. **Query Translation (HyDE)**: Expanding questions into hypothetical answers and sub-queries.
5. **Hybrid Fusion (RRF) Retrieval**: Blending dense vector retrieval and sparse BM25 keyword matching with Reciprocal Rank Fusion.
6. **Context Reranking**: Compressing candidate nodes down to the absolute best contexts to minimize generation cost.
7. **Ragas Scorecard Evaluation**: Running live evaluations measuring Faithfulness, Answer Relevance, Context Precision, and Context Recall.
8. **Dynamic Interactive Search Terminal**: Dynamic ipywidgets Search GUI.


In [ ]:
# Cell 1: Environment & Dependency Setup
import sys
import os

# Check if running inside Google Colab
is_colab = 'google.colab' in sys.modules

if is_colab:
    print("Detected Google Colab environment. Running package auto-installation...")
    !pip install -q llama-index llama-index-vector-stores-chroma llama-index-graph-stores-neo4j llama-index-retrievers-bm25 ragas chromadb pypdf ipywidgets nest_asyncio pandas langchain-openai
    print("\nAll dependencies successfully installed!")
else:
    print("Running locally. Please ensure the workspace virtualenv is active and packages are installed.")

# Apply nest_asyncio to support nested event loops in Jupyter notebooks (required by Ragas)
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# Cell 2: OpenAI API Configuration Widget (Colab-Friendly)
import ipywidgets as widgets
from IPython.display import display

# Input widget for API key
api_key_input = widgets.Password(
    value=os.getenv("OPENAI_API_KEY", ""),
    placeholder='Enter your OpenAI API Key...',
    description='OpenAI Key:',
    disabled=False,
    layout=widgets.Layout(width='500px')
)

print("Please check/input your OpenAI API Key below:")
display(api_key_input)


In [ ]:
# Cell 3: Local Hot-Folder Ingestion Directories Setup
from pathlib import Path

# Configure directories
DATA_DIR = Path("./rag_data")
NEED_PROCESSING = DATA_DIR / "need-processing"
PROCESSED = DATA_DIR / "processed"

NEED_PROCESSING.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Ingestion folders initialized successfully:")
print(f"- 📂 Upload items here: {NEED_PROCESSING.resolve()}")
print(f"- 📂 Completed items move to: {PROCESSED.resolve()}")


In [ ]:
# Cell 4: Pedagogical Document File Uploader Widget
uploader = widgets.FileUpload(
    accept='.pdf,.txt,.md',
    multiple=False,
    description="Upload File"
)

def on_upload_change(change):
    if not uploader.value:
        return

    # ipywidgets 8.x (Colab default): .value is a tuple of dicts
    uploaded_file = uploader.value[0]
    filename = uploaded_file['name']
    content = bytes(uploaded_file['content'])

    target_path = NEED_PROCESSING / filename
    with open(target_path, 'wb') as f:
        f.write(content)

    print(f"\n[Uploaded] '{filename}' successfully saved to {target_path.name}!")

uploader.observe(on_upload_change, names='value')
print("Click below to upload a PDF, TXT, or MD document into the hot-folder:")
display(uploader)


In [ ]:
# Cell 5: Source Classification & Adaptive Chunking strategies
import re
import shutil
from llama_index.core import Document
from llama_index.core.node_parser import (
    HTMLNodeParser,
    HierarchicalNodeParser,
    MarkdownNodeParser,
    SemanticSplitterNodeParser,
    SentenceSplitter,
    TokenTextSplitter,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import SimpleDirectoryReader

SUPPORTED_SUFFIXES = {".pdf", ".txt", ".md"}

def detect_kind(text: str) -> tuple[str, str]:
    lowered = text.lower()
    if "<html" in lowered or "<div" in lowered:
        return "html_page", "html"
    if "# " in text or "## " in text or "- " in text:
        return "markdown_notes", "markdown"
    if re.search(r"\b\d{1,2}:\:\d{2}\b", text) or "speaker" in lowered:
        return "transcript", "semantic"
    if re.search(r"\b(error|warn|trace|stack|http)\b", lowered):
        return "unstructured_logs", "token"
    if len(text.splitlines()) > 8 and len(text.split()) > 180:
        return "reference_report", "hierarchical"
    return "clean_text", "sentence"

def build_parser(strategy: str):
    os.environ["OPENAI_API_KEY"] = api_key_input.value
    embed_model = OpenAIEmbedding(model="text-embedding-3-small")

    if strategy == "markdown":
        return MarkdownNodeParser()
    if strategy == "html":
        return HTMLNodeParser()
    if strategy == "semantic":
        return SemanticSplitterNodeParser(
            embed_model=embed_model,
            breakpoint_percentile_threshold=90,
        )
    if strategy == "hierarchical":
        return HierarchicalNodeParser.from_defaults(chunk_sizes=[1536, 768, 256])
    if strategy == "token":
        return TokenTextSplitter(chunk_size=512, chunk_overlap=50)
    return SentenceSplitter(chunk_size=768, chunk_overlap=80)

def ingest_and_parse():
    os.environ["OPENAI_API_KEY"] = api_key_input.value
    # Filter to supported file types only, skip hidden files and directories
    files = sorted(
        p for p in NEED_PROCESSING.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES
    )
    if not files:
        print("⚠️ No files found inside 'need-processing' hot-folder yet. Please upload a file in Cell 4.")
        return []

    documents = SimpleDirectoryReader(input_files=[str(f) for f in files]).load_data()
    all_nodes = []

    for path, doc in zip(files, documents):
        text = doc.get_content()
        category, strategy = detect_kind(text)
        print(f"📂 Processing '{path.name}' -> Category: {category} (Strategy: {strategy})")

        parser = build_parser(strategy)
        nodes = parser.get_nodes_from_documents([doc])
        for node in nodes:
            node.metadata.update({
                "title": path.name,
                "url": str(path.resolve()),
                "category": category,
                "chunk_strategy": strategy
            })
        all_nodes.extend(nodes)

        # Clean up hot-folder after parsing is complete
        shutil.move(str(path), str(PROCESSED / path.name))

    print(f"\nIngestion complete: {len(files)} files split into {len(all_nodes)} dynamic nodes!")
    return all_nodes

nodes = ingest_and_parse()


In [ ]:
# Cell 6: ChromaDB Vector Store & SimpleGraph Store Ingestion
import chromadb
from llama_index.core import StorageContext, VectorStoreIndex, KnowledgeGraphIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.core import Settings

# Global index hooks
vector_index = None
graph_index = None

def build_databases(nodes_list):
    global vector_index, graph_index
    if not nodes_list:
        print("⚠️ Ingested nodes list is empty. Please run Cell 5 successfully.")
        return

    os.environ["OPENAI_API_KEY"] = api_key_input.value
    Settings.llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0.2)
    Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

    # Initialize Chroma persistent DB
    db_path = DATA_DIR / "chroma_db"
    chroma_client = chromadb.PersistentClient(path=str(db_path))
    chroma_collection = chroma_client.get_or_create_collection("colab_pedagogical_rag")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

    # Initialize SimpleGraph local store
    graph_store = SimpleGraphStore()

    storage_context = StorageContext.from_defaults(
        vector_store=vector_store,
        graph_store=graph_store
    )

    # Batch-embed and insert using single IngestionPipeline
    pipeline = IngestionPipeline(transformations=[Settings.embed_model])
    processed_nodes = list(pipeline.run(nodes=nodes_list))

    print("💾 Feeding embedded nodes into ChromaDB vector index...")
    vector_index = VectorStoreIndex(
        processed_nodes,
        storage_context=storage_context
    )

    print("💾 Extracting triplets and building Knowledge Graph index...")
    try:
        docs = [Document(text=n.get_content(), metadata=n.metadata) for n in nodes_list]
        graph_index = KnowledgeGraphIndex.from_documents(
            docs,
            storage_context=storage_context,
            max_triplets_per_chunk=2,
            include_embeddings=True,
            show_progress=False
        )
        print("💾 SQLite ChromaDB and local Knowledge Graph created successfully!")
    except Exception as e:
        print(f"⚠️ SimpleGraph triplets extraction fallback: {str(e)}")
        graph_index = None

build_databases(nodes)


In [ ]:
# Cell 7: Advanced Query Translation (HyDE) Inspector
from llama_index.core.indices.query.query_transform import HyDEQueryTransform

def inspect_query_translation(question: str):
    os.environ["OPENAI_API_KEY"] = api_key_input.value
    transform = HyDEQueryTransform(llm=Settings.llm, include_original=True)
    query_bundle = transform.run(question)

    print(f"🔎 Original Query: '{question}'\n")
    print(f"🔎 Generated Hypothetical Answer (HyDE):\n{query_bundle.custom_embedding_strs[0]}\n")
    print("🔎 Sub-Queries generated for dense search list:")
    for i, q in enumerate(query_bundle.embedding_strs):
        print(f"  - Query {i+1}: {q}")
    return query_bundle

test_question = "Tell me about the design features and target audience of the document."
if vector_index:
    query_bundle = inspect_query_translation(test_question)
else:
    print("⚠️ Please index a document in Cell 6 first.")


In [ ]:
# Cell 8: Hybrid Fusion (RRF) & KG Graph Traversal
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.indices.knowledge_graph.retrievers import KGTableRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.postprocessor import LLMRerank

def retrieve_hybrid_and_graph(question: str):
    os.environ["OPENAI_API_KEY"] = api_key_input.value
    query_bundle = inspect_query_translation(question)

    # 1. Dense Vector Retriever
    vector_retriever = vector_index.as_retriever(similarity_top_k=6)

    # 2. Sparse BM25 Keyword Retriever
    nodes_list = list(vector_index.docstore.docs.values())
    bm25_retriever = BM25Retriever.from_defaults(nodes=nodes_list, similarity_top_k=6)

    # 3. Blended Fusion via Reciprocal Rank Fusion (RRF)
    fusion_retriever = QueryFusionRetriever(
        [vector_retriever, bm25_retriever],
        llm=Settings.llm,
        num_queries=3,
        mode="reciprocal_rerank",
        similarity_top_k=5,
        use_async=False
    )

    candidates = list(fusion_retriever.retrieve(query_bundle))
    print(f"\n🎯 Dense-Sparse candidates retrieved: {len(candidates)}")

    # 4. Multi-hop Graph Triplet lookup
    if graph_index:
        try:
            graph_retriever = KGTableRetriever(graph_index, retriever_mode="keyword")
            graph_nodes = graph_retriever.retrieve(question)
            print(f"🎯 Multi-hop Entity Graph nodes retrieved: {len(graph_nodes)}")
            candidates.extend(graph_nodes)
        except Exception:
            pass

    # Deduplicate candidates
    unique = {}
    for item in candidates:
        unique[item.node.node_id] = item
    deduped_candidates = list(unique.values())
    print(f"🎯 Total unique merged candidates: {len(deduped_candidates)}")

    # 5. Reranking and Context Pruning
    print("\n🎯 Pruning context using LLMRerank...")
    reranker = LLMRerank(llm=Settings.llm, choice_batch_size=5, top_n=2)
    reranked = reranker.postprocess_nodes(deduped_candidates, query_bundle=query_bundle)

    print(f"\n--- Context Pruning Complete: Top 2 Nodes Selected ---")
    for idx, node_item in enumerate(reranked):
        title = node_item.node.metadata.get('title', 'unknown')
        excerpt = node_item.node.get_content()[:200]
        print(f"\n[Rank {idx+1}] Score: {node_item.score:.2f} | Source: {title}")
        print(f"Excerpt: {excerpt}...")

    return reranked

if vector_index:
    reranked_nodes = retrieve_hybrid_and_graph(test_question)


In [ ]:
# Cell 9: Ragas Scorecard dynamic evaluator
import asyncio
from ragas import SingleTurnSample
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pandas as pd

def run_ragas_evaluation(question: str, answer: str, contexts: list, reference: str):
    os.environ["OPENAI_API_KEY"] = api_key_input.value

    # Initialize evaluation wraps
    llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0.0))
    embeds = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

    sample = SingleTurnSample(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts,
        reference=reference
    )

    metrics = {
        "Faithfulness": Faithfulness(llm=llm),
        "Answer Relevance": ResponseRelevancy(llm=llm, embeddings=embeds),
        "Context Precision": LLMContextPrecisionWithoutReference(llm=llm),
        "Context Recall": LLMContextRecall(llm=llm)
    }

    print("\n--- Running Ragas LLM-as-a-judge Scorecard Evaluation ---")
    scores = {}
    loop = asyncio.get_event_loop()
    for name, metric in metrics.items():
        score = loop.run_until_complete(metric.single_turn_ascore(sample))
        scores[name] = round(float(score), 4)

    df = pd.DataFrame(list(scores.items()), columns=["Ragas Metric", "Score [0.0 - 1.0]"])
    display(df.style.background_gradient(cmap="RdYlGn", subset=["Score [0.0 - 1.0]"]))
    return scores


In [ ]:
# Cell 10: Dynamic RAG Search GUI Terminal
from IPython.display import display, clear_output

# Design layout widgets
query_input = widgets.Text(
    value='',
    placeholder='Ask a search question here...',
    description='Question:',
    layout=widgets.Layout(width='75%')
)

search_button = widgets.Button(
    description='Search RAG',
    button_style='info',
    icon='search'
)

output_area = widgets.Output()

def execute_pipeline(change):
    with output_area:
        clear_output()
        question = query_input.value.strip()
        if not question:
            print("⚠️ Please type a question in the search input.")
            return

        if not vector_index:
            print("⚠️ Please upload and index a document first in Cells 4, 5, and 6.")
            return

        print(f"🚀 Executing complete hybrid retrieval pipeline for: '{question}'...")

        # 1. Retrieve Reranked Nodes
        nodes = retrieve_hybrid_and_graph(question)
        contexts = [node_item.node.get_content() for node_item in nodes]

        # 2. Synthesize Answer
        context_block = "\n\n".join(contexts)
        prompt = f"""Answer the question using ONLY the provided contexts.
        Indicate which source document the fact was retrieved from.

        Question: {question}

        Contexts:
        {context_block}"""

        answer = Settings.llm.complete(prompt).text.strip()

        # 3. Create Ragas Reference Summary
        ref_prompt = f"Summarize the context facts accurately to answer: '{question}'"
        reference = Settings.llm.complete(ref_prompt).text.strip()

        print("\n=========================================================================")
        print("🧠 SYNTHESIZED RAG ANSWER")
        print("=========================================================================")
        print(answer)
        print("=========================================================================")

        # 4. Run Ragas Scorecard Evaluation
        run_ragas_evaluation(question, answer, contexts, reference)

search_button.on_click(execute_pipeline)
layout_box = widgets.HBox([query_input, search_button])
display(widgets.VBox([layout_box, output_area]))
